# Baseline vs. Attention-Entropy Regularization

This notebook compares a baseline Kepler block-size run with a run trained using normalized causal-attention entropy regularization. Update the configuration below to match the two `.npz` files produced by `kepler_cv_blocksize.py`.

The regularizer is active on the half-open step interval `[start_step, end_step)`. Thus, `start_step=1000, end_step=None` means **after step 1000**, while `start_step=0, end_step=1000` means **before step 1000**.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
RESULT_DIR = Path('./results/kepler_cv_blocksize')


## Experiment files

In [ ]:
# Keep these values synchronized with kepler_cv_blocksize.py.
block_size = 100
num_trajectories = 10_000
noise_scale = 0.1
loss_mask = 'all'

# Entropy experiment settings. Positive coefficients penalize diffuse attention.
entropy_coefficient = 1e-3
entropy_start_step = 1000       # inclusive; use 0 for regularization from the beginning
entropy_end_step = None         # exclusive; None means through the final training step

base_stem = (
    f'results_block_size_{block_size}_num_trajectories_{num_trajectories}'
    f'_noise_scale_{noise_scale}_loss_mask_{loss_mask}'
)
end_label = 'none' if entropy_end_step is None else str(entropy_end_step)
entropy_suffix = (
    f'_attention_entropy_reg_{entropy_coefficient}'
    f'_start_{entropy_start_step}_end_{end_label}'
)

baseline_path = RESULT_DIR / f'{base_stem}.npz'
entropy_path = RESULT_DIR / f'{base_stem}{entropy_suffix}.npz'

print('Baseline:', baseline_path)
print('Entropy: ', entropy_path)
assert baseline_path.exists(), f'Missing baseline result: {baseline_path}'
assert entropy_path.exists(), f'Missing entropy result: {entropy_path}'


## Load and inspect results

In [ ]:
def load_result(path):
    with np.load(path, allow_pickle=True) as loaded:
        result = {}
        for key in loaded.files:
            value = loaded[key]
            # Unwrap scalar object arrays while leaving numeric histories as arrays.
            result[key] = value.item() if value.shape == () else value
    return result

baseline = load_result(baseline_path)
entropy_run = load_result(entropy_path)

summary_keys = [
    'block_size', 'num_trajectories', 'noise_scale', 'loss_mask',
    'attention_entropy_reg', 'attention_entropy_reg_start_step',
    'attention_entropy_reg_end_step', 'final_test_loss'
]
pd.DataFrame({
    'baseline': {k: baseline.get(k, 'not recorded') for k in summary_keys},
    'entropy_regularized': {k: entropy_run.get(k, 'not recorded') for k in summary_keys},
})


## Training and test losses

In [ ]:
def prediction_history(result):
    # Old baseline files do not have prediction_losses; train_losses is equivalent
    # because their entropy coefficient was zero.
    return np.asarray(result.get('prediction_losses', result['train_losses']), dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for label, result, color in [
    ('Baseline', baseline, 'tab:blue'),
    ('Entropy regularized', entropy_run, 'tab:orange'),
]:
    axes[0].plot(prediction_history(result), label=label, color=color, alpha=0.9)
    axes[1].plot(np.asarray(result['test_losses'], dtype=float), label=label, color=color, alpha=0.9)

axes[0].set(title='Prediction loss (train)', xlabel='Training step', ylabel='MSE', yscale='log')
axes[1].set(title='Test loss', xlabel='Training step', ylabel='MSE', yscale='log')
for ax in axes:
    ax.legend()
plt.tight_layout()


## Attention entropy and active regularization interval

In [ ]:
entropies = np.asarray(entropy_run['attention_entropies'], dtype=float)
reg_losses = np.asarray(entropy_run['attention_entropy_reg_losses'], dtype=float)
active = np.asarray(entropy_run['attention_entropy_reg_active'], dtype=bool)
steps = np.arange(len(entropies))

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].plot(steps, entropies, color='tab:purple', label='Normalized attention entropy')
axes[0].set_ylabel('Entropy (0–1)')
axes[0].set_title('Attention entropy during training')
axes[1].plot(steps, reg_losses, color='tab:red', label='Weighted entropy regularizer')
axes[1].set(xlabel='Training step', ylabel='Regularization loss', title='Entropy contribution to total loss')

if active.any():
    active_steps = steps[active]
    for ax in axes:
        ax.axvspan(active_steps.min(), active_steps.max() + 1, color='tab:orange', alpha=0.15, label='Regularizer active')
for ax in axes:
    ax.legend()
plt.tight_layout()


## Periodic rollout error comparison

In [ ]:
def evaluation_history(result, split='test'):
    steps = np.asarray(result['eval_steps'], dtype=int)
    evaluations = list(result['eval_results'])
    errors = np.asarray([
        evaluation[f'error_stats_{split}']['mean_error'] for evaluation in evaluations
    ], dtype=float)
    return steps, errors

fig, ax = plt.subplots(figsize=(8, 4.5))
for label, result, color in [
    ('Baseline', baseline, 'tab:blue'),
    ('Entropy regularized', entropy_run, 'tab:orange'),
]:
    eval_steps, errors = evaluation_history(result, 'test')
    ax.plot(eval_steps, errors, marker='o', markersize=3, label=label, color=color)
ax.set(xlabel='Training step', ylabel='Mean rollout position error', title='Test rollout error')
ax.set_yscale('log')
ax.legend()
plt.tight_layout()


## Final linear-probe comparison

In [ ]:
def final_probe_frame(result, probe_key='probe_results', score='r2_last'):
    final_eval = list(result['eval_results'])[-1]
    probes = final_eval.get(probe_key)
    if not probes:
        return pd.DataFrame()
    return pd.DataFrame({
        layer: {target: values[score] for target, values in targets.items()}
        for layer, targets in probes.items()
    })

baseline_probes = final_probe_frame(baseline)
entropy_probes = final_probe_frame(entropy_run)
probe_comparison = pd.DataFrame({
    'baseline_mean_r2_last': baseline_probes.mean(axis=1),
    'entropy_mean_r2_last': entropy_probes.mean(axis=1),
}).sort_index()
probe_comparison['difference'] = (
    probe_comparison['entropy_mean_r2_last'] - probe_comparison['baseline_mean_r2_last']
)

ax = probe_comparison[['baseline_mean_r2_last', 'entropy_mean_r2_last']].plot.bar(
    figsize=(14, 5), color=['tab:blue', 'tab:orange']
)
ax.set(xlabel='Probe target', ylabel='Mean R² across layers', title='Final linear-probe performance')
ax.axhline(0, color='black', linewidth=0.8)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
probe_comparison


## Final numerical summary

In [ ]:
def final_rollout_error(result, split='test'):
    return float(list(result['eval_results'])[-1][f'error_stats_{split}']['mean_error'])

summary = pd.DataFrame({
    'baseline': {
        'final prediction loss': prediction_history(baseline)[-1],
        'final test loss': np.asarray(baseline['test_losses'], dtype=float)[-1],
        'final test rollout error': final_rollout_error(baseline),
    },
    'entropy_regularized': {
        'final prediction loss': prediction_history(entropy_run)[-1],
        'final test loss': np.asarray(entropy_run['test_losses'], dtype=float)[-1],
        'final test rollout error': final_rollout_error(entropy_run),
        'final normalized attention entropy': entropies[-1],
    },
})
summary['relative_change'] = summary['entropy_regularized'] / summary['baseline'] - 1
summary
